In [ ]:
%pylab inline
import os
import json
import eucare as ec
from eucare.half import Vertex, HalfEdge, Face

In [ ]:
G = ec.io.load_graph('./test_save.heg')
G.show()

In [ ]:
FILE_CLASSES = [
    'singleModel', 
    'multiModel', 
    'animation', 
    'diagrams'
]


FRAME_CLASSES = [
    'creasePattern',
    'foldedForm',
    'graph',
    'linkage'
]


FRAME_ATTRIBUTES = [
    '2D', 
    '3D', 
    'abstract', 
    'manifold', 
    'nonManifold', 
    'orientable', 
    'nonOrientable', 
    'selfTouching', 
    'nonSelfTouching', 
    'selfIntersecting',
    'nonSelfIntersecting'
]


FRAME_UNITS = [
    'unit',
    'in',
    'pt',
    'm',
    'cm',
    'mm',
    'um',
    'nm'
]

EDGES_ASSIGNMENT = [
    'B',  # border/boundary edge (only one incident face)
    'M',  # mountain fold
    'V',  # valley folde
    'F',  # flat (unfolded) fold
    'U'   # unassigned/unknown
]


def save_fold(
    filename, G, overwrite=False, author=None, 
    file_title=None, file_description=None, file_classes=None,
    frame_classes='graph', frame_attributes=None, frame_unit=None
#     attributes_to_save=('pos', 'length', 'in_angle', 'color_key')
):
    if not filename.endswith('.fold'):
        filename += '.fold'
    assert overwrite or not os.path.exists(filename), \
        f'File exists: {filename}. Specify overwrite=True to overwrite.'
    
    # file attributes
    file_dict = dict(
        file_spec='1.1',
        file_creator='eucare'
    )
    if author is not None:
        fold_dict['file_author'] = author
    if file_title is not None:
        fold_dict['file_title'] = file_title
    if file_description is not None:
        fold_dict['file_description'] = file_description
    if file_classes is not None:
        if isinstance(file_classes, str):
            file_classes = [file_classes]
        for item in file_classes:
            assert item in FILE_CLASSES or ':' in item, \
                f'Custom classes should include ":". Standard classes are {FILE_CLASSES}. Got {item}.'
        fold_dict['file_classes'] = classes
    
    # frame attributes
    frame_dict = dict()
    if frame_classes is not None:
        if isinstance(frame_classes, str):
            frame_classes = [frame_classes]
        for item in frame_classes:
            assert item in FRAME_CLASSES or ':' in item, \
                f'Custom classes should include ":". Standard classes are {FRAME_CLASSES}. Got {item}.'
        frame_dict['frame_classes'] = frame_classes
    if frame_attributes is not None:
        if isinstance(frame_attributes, str):
            frame_attributes = [frame_attributes]
        for item in frame_attributes:
            assert item in FRAME_ATTRIBUTES or ':' in item, \
                f'Custom attributes should include ":". Standard attributes are {FRAME_ATTRIBUTES}. Got {item}.'
        frame_dict['frame_attributes'] = frame_attributes
    if frame_unit is not None:
        assert frame_unit in FRAME_UNITS, f'Got unknown unit {frame_unit}. Select one of {FRAME_UNITS}.'
    
    # vertex, edge and face orders
    vs = list(G.vertices)
    es = G.halfedges_representing_edges()
    fs = list(G.faces)
    v_index = {v: i for i, v in enumerate(vs)}
    e_index = {e: i for i, e in enumerate(es)}
    e_index.update({e.rev: i for e, i in e_index.items()})
    f_index = {f: i for i, f in enumerate(fs)}
    
    # vertices
    frame_dict['vertices_coords'] = [v['pos'].tolist() for v in v_index.keys()]
    frame_dict['vertices_vertices'] = [[v2['pos'].tolist() for v2 in v.vertex_iter()] for v in vs]
    
    # edges
    frame_dict['edges_vertices'] = [[v_index[e.orig], v_index[e.dest]]for e in es]
    
    def edge_to_faces(e):
        # first face to the left, then to the right
        result = []
        if e.face is not None:
            result.append(f_index[e.face])
        if e.rev.face is not None:
            result.append(f_index[e.rev.face])
        return result
    
    frame_dict['edges_faces'] = [edge_to_faces(e) for e in es]
    
    def edge_assignment(e):
        if e.on_border() or e.rev.on_border():
            return 'B'
        return e.attributes.get('edge_assignment', e.rev.attributes.get('edge_assignment', 'U'))
    
    frame_dict['edges_assignment'] = [edge_assignment(e) for e in es]
    
    # TODO: implement edges_foldAngle for 3D models
    
    # faces
    frame_dict['faces_vertices'] = [[v_index[v] for v in f.vertex_iter()] for f in fs]
    frame_dict['faces_edges'] = [[e_index[e] for e in f.halfedge_iter()] for f in fs]
    
    # TODO: implement faceOrders for folded models
    # TODO: understand what edgeOrders means
    
    fold_dict = file_dict.copy()
    fold_dict.update(frame_dict)
    with open(filename, 'w') as f:
        f.write(json.dumps(fold_dict, indent=4))


G.recompute_lengths_and_angles()
save_fold('test_fold.fold', G, overwrite=True)

In [ ]:
with open('test.json', 'w') as f:
    json.dump(dict(a=3), f)

In [ ]:
with open('test_fold2.fold', 'r') as f:
    print(f.read())

In [ ]:
d = {'a': {'a': 3, 'b': 4}}
print(json.dumps(d, indent=4))